# Milestone 6-D — 멀티모델 검증 (TinyLlama-1.1B)

mini_compressor가 Qwen3 외 아키텍처에서도 동작하는지 검증한다.

타깃: `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — `model_type=llama` (LLaMA 아키텍처, RMSNorm, GQA).

핵심: **라이브러리 코드를 한 줄도 수정하지 않고** recipe가 동작해야 한다.

In [1]:
import sys
sys.path.insert(0, "..")

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

from mini_compressor import Compressor
from mini_compressor.modifiers.smoothquant import _find_smooth_pairs
from mini_compressor.fake_quant_linear import FakeQuantLinear

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT = "The key advantage of quantization is"
print("device:", DEVICE)

/home/cyh/projects/mini-compressor/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 1. 모델 구조 — Qwen3와 다른 아키텍처 확인

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
cfg = AutoConfig.from_pretrained(MODEL_ID)
print(f"model_type    = {cfg.model_type}")
print(f"layers        = {cfg.num_hidden_layers}")
print(f"attn/kv heads = {cfg.num_attention_heads}/{cfg.num_key_value_heads}  -> GQA {cfg.num_attention_heads}:{cfg.num_key_value_heads}")

def load_model():
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
    m.eval()
    return m

model_type    = llama
layers        = 22
attn/kv heads = 32/4  -> GQA 32:4


## 2. SmoothQuant pair 자동 탐색

`_find_smooth_pairs`는 `input_layernorm`/`post_attention_layernorm` 이름으로 norm-linear 페어를 찾는다.
LLaMA도 이 이름 규칙을 따르므로 수정 없이 동작해야 한다.

In [3]:
model = load_model()
pairs = _find_smooth_pairs(model)
print(f"SmoothQuant pairs: {len(pairs)}  (= layers x 2 = {cfg.num_hidden_layers * 2})")

norm, linears = pairs[0]
print(f"\nattn pair  norm = {type(norm).__name__}")
for l in linears:
    print(f"  Linear(in={l.in_features}, out={l.out_features})")
print("  -> q_proj out > k/v_proj out: GQA 구조가 페어에 그대로 반영됨")
del model
torch.cuda.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/201 [00:00<00:21,  9.52it/s]

Loading weights:  16%|█▌        | 32/201 [00:00<00:01, 150.60it/s]

Loading weights:  29%|██▉       | 58/201 [00:00<00:00, 191.33it/s]

...

Loading weights:  66%|██████▌   | 132/201 [00:00<00:00, 221.20it/s]

Loading weights:  83%|████████▎ | 166/201 [00:00<00:00, 249.22it/s]

Loading weights:  96%|█████████▌| 193/201 [00:00<00:00, 249.08it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 220.91it/s]

SmoothQuant pairs: 44  (= layers x 2 = 44)

attn pair  norm = LlamaRMSNorm
  Linear(in=2048, out=2048)
  Linear(in=2048, out=256)
  Linear(in=2048, out=256)
  -> q_proj out > k/v_proj out: GQA 구조가 페어에 그대로 반영됨


## 3. recipe end-to-end — compress → generate

`from_recipe` 네 가지 preset을 순회하며 compress 후 generate. `targets`/`ignore`는 model-agnostic.

In [4]:
calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Quantization reduces model size by representing weights in lower precision.",
    "Large language models require significant computational resources.",
]
calib = [{k: v.to(DEVICE) for k, v in tokenizer(t, return_tensors="pt").items()} for t in calib_texts]

def generate(model):
    inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for recipe in ["w4a16", "w8a8", "w8a8_dynamic", "w8a8_smoothquant"]:
    model = load_model()
    dl = calib if recipe in ("w8a8", "w8a8_smoothquant") else None
    Compressor.from_recipe(recipe, ignore=["lm_head"]).compress(model, dataloader=dl)
    fql = sum(1 for m in model.modules() if isinstance(m, FakeQuantLinear))
    print(f"[{recipe:18s}] FakeQuantLinear={fql}  lm_head={type(model.lm_head).__name__}")
    print(f"  {generate(model)}\n")
    del model
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  11%|█         | 22/201 [00:00<00:00, 204.61it/s]

Loading weights:  29%|██▉       | 58/201 [00:00<00:00, 284.37it/s]

Loading weights:  47%|████▋     | 94/201 [00:00<00:00, 298.89it/s]

Loading weights:  62%|██████▏   | 124/201 [00:00<00:00, 295.34it/s]

...

Loading weights:  88%|████████▊ | 176/201 [00:01<00:00, 154.44it/s]

Loading weights: 100%|██████████| 201/201 [00:01<00:00, 182.95it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[w8a8_smoothquant  ] FakeQuantLinear=154  lm_head=Linear


  The key advantage of quantization is that it can be used to reduce the number of bits required for the representation of a digital signal. This is because the quantization process can be used



## 결론

TinyLlama-1.1B (LLaMA 아키텍처)에서 **mini_compressor 라이브러리 코드를 한 줄도 수정하지 않고** 네 가지 recipe가 모두 동작했다.

- `model_type=llama` — Qwen3와 다른 아키텍처
- SmoothQuant `_find_smooth_pairs`가 RMSNorm 기반으로 44개 페어를 자동 탐색 — GQA 구조도 그대로 반영
- `targets`/`ignore` 패턴이 model-agnostic — `lm_head`만 제외, 나머지 Linear 교체
- end-to-end (compress → generate)가 architecture-specific 오류 없이 완료

`python eval.py --model <id>`로 임의 모델에 동일 흐름을 재현할 수 있다.